# BEiT image logits for fusion

Extract **test-set** logits from the fine-tuned BEiT-Base/p16 checkpoint. Run for `variant` = `"mini"` and `"full"`.

In [ ]:
import os
import cv2
import numpy as np
import torch
import timm
import torchvision.transforms as T
from PIL import Image, ImageFile
from tqdm import tqdm

from vlm_utils import VARIANTS

ImageFile.LOAD_TRUNCATED_IMAGES = True

dataset_dir = '/FungiTastic'
variant = 'mini'  # 'mini' | 'full'
device = 'cuda'
model_name = f'beit-224-{variant}'
model_id = 'hf-hub:BVRA/beit_base_patch16_224.in1k_ft_fungitastic_224'

cfg = VARIANTS[variant]
split_name = f'{cfg["beit_prefix"]}-test'
image_folder = os.path.join(dataset_dir, cfg['test_images'])

transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class ImageFolderDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms, fallback=None):
        self.files = sorted(os.listdir(root))
        self.root = root
        self.transforms = transforms
        self.fallback = fallback

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        path = os.path.join(self.root, name)
        try:
            img = Image.open(path).convert('RGB')
        except (OSError, Image.UnidentifiedImageError):
            img_cv = cv2.imread(path)
            if img_cv is None:
                if self.fallback is None:
                    raise
                return self.fallback.clone(), name
            img = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
        return self.transforms(img), name


model = timm.create_model(model_id, pretrained=True).to(device).eval()

In [ ]:
os.makedirs(f'features/{model_name}', exist_ok=True)

if not os.path.isdir(image_folder):
    raise FileNotFoundError(image_folder)

dataset = ImageFolderDataset(image_folder, transforms, fallback=torch.randn(3, 224, 224))
loader = torch.utils.data.DataLoader(dataset, batch_size=64, num_workers=6, shuffle=False)

logits_list, files_list = [], []
for imgs, files in tqdm(loader, desc=split_name):
    with torch.no_grad():
        logits_list.append(model(imgs.to(device)).cpu())
        files_list.append(np.array(files))

bundle = {
    'files': np.concatenate(files_list),
    'logits': torch.cat(logits_list),
}
out_path = f'features/{model_name}/{split_name}.pth'
torch.save(bundle, out_path)
print('Saved', out_path, bundle['logits'].shape)